### Dataset Overview 

- Study period: **2006-01 – 2025-12**, aligned to **Friday-ending weekly frequency (W-FRI)**.
- All sources are public and free; only Global Fishing Watch (M3) requires a free API token.

**M1) Core Market, Macro & Financial Variables**

10 literature-core variables (Kilian supply / global-demand / precautionary-demand mechanisms plus market-financial conditions), together with extended fundamentals, macro controls and derived columns. The raw layer is organised **by data provider** — `EIA/`, `FRED/`, `Yahoo/`, `Other/` — acquired or register-only through a single entry point `download_m1_raw.py` with one audit `manifest.csv` (per-file provider, source URL, identifier, download time, rows, coverage, SHA-256). It is then built **offline** by `build_m1_weekly.py` into one weekly feature table `m1_weekly_features.csv` (**35 columns**, all peers; no intermediate table or separate merge step; Brent/WTI/S&P 500 all as log returns). All aligned to Friday-ending weekly frequency (W-FRI), January 2006 – December 2025; the standardised M0–M4 comparison window is **2019–2026** (clipped at modelling time, to align with M2/M3).

| Dataset | Summary | URL | Variables | Time Range | Frequency | Coverage | Data Type | Access | Potential Use | Limitations |
|---------|---------|-----|-----------|------------|-----------|----------|-----------|--------|---------------|-------------|
| **EIA/** — Brent & WTI Spot Prices + Weekly Petroleum Status Report (WPSR) | Official EIA Brent and WTI crude spot prices plus US weekly petroleum fundamentals (inventories, production, trade, refinery runs, product supplied); manually exported `.xls`, register-only into the manifest | [EIA Brent](https://www.eia.gov/petroleum/gasdiesel/); [EIA WPSR](https://www.eia.gov/petroleum/supply/weekly/) | Brent & WTI spot price (next-week Brent price is the sole target), lagged prices, log returns (training target, price reconstructed), Brent–WTI spread; weekly commercial crude stocks (excl. SPR) and first-difference `crude_stocks_change`, Cushing stocks (+change), crude production, imports, exports (imports/exports kept separately; net-trade column removed), refinery crude input and utilisation; **product supplied of finished motor gasoline, distillate fuel oil, and kerosene-type jet fuel** (demand proxies) | Jan 2006–Dec 2025 | Daily (prices) and weekly (WPSR); aggregated to W-FRI | Europe Brent & US WTI benchmarks; United States (fundamentals) | Price / inventory / demand-proxy data | Open (manual EIA export) | Oil-price momentum, lag features and the price target (trained via log return; direction derived from the predicted price for evaluation, no volatility forecast); supply-demand balance via inventory build/draw; refinery activity and product-supplied demand proxies (road, freight/heating, aviation) | US-only fundamentals; product supplied is an implied demand proxy with strong seasonality; a single Brent series does not capture the global crude balance |
| **Other/** — Dallas Fed Index of Global Real Economic Activity (Kilian REA) | Kilian's index of global real economic activity based on dry cargo shipping rates, used as a proxy for global industrial commodity demand (`global_econ_activity`) | [Dallas Fed](https://www.dallasfed.org/research/igrea) | Global real economic activity index | Feb 2006–Dec 2025 | Monthly; month-end forward-filled and lagged ~5 weeks to weekly to respect publication delay | Global | Macro activity index | Open (auto-download) | Proxy for aggregate global demand for industrial commodities, following the Kilian (2009) demand identification framework | Monthly frequency limits short-term responsiveness; publication lag of approximately five weeks |
| **FRED/** — IMF Industrial Materials Index, 10-Year Treasury, VIX, Broad Dollar Index, Fed Funds Rate | Macro-financial control indicators downloaded via FRED | [PINDUINDEXM](https://fred.stlouisfed.org/series/PINDUINDEXM); [DGS10](https://fred.stlouisfed.org/series/DGS10); [VIXCLS](https://fred.stlouisfed.org/series/VIXCLS); [DTWEXBGS](https://fred.stlouisfed.org/series/DTWEXBGS); [DFF](https://fred.stlouisfed.org/series/DFF) | Global price index of industrial materials; 10-year Treasury yield change (first difference, `dgs10_change`); CBOE VIX close; broad trade-weighted US dollar index; effective federal funds rate | Jan 2006–Dec 2025 (industrial materials from Feb 2006) | Daily (Treasury, VIX, dollar index, funds rate) and monthly (industrial materials); aggregated to weekly | US financial markets and global commodity markets | Interest rate / volatility / FX / commodity-price indices | Open | Control for non-oil industrial demand, interest-rate changes and holding costs (Δ10Y, not the level, which fails a unit-root test), broad financial-market uncertainty, and US dollar conditions | US-centric proxies; VIX reflects equity-market rather than oil-specific uncertainty; broad dollar index evidence is limited (commodity-exporter FX preferred for the core); monthly industrial materials index introduces publication lag |
| **Yahoo Finance/** — S&P 500, CBOE OVX, Brent Front-Month Futures, Commodity Currencies, Gold | Market-traded equity index, oil-specific volatility index, oil futures, commodity-exporter exchange rates, and gold price | [^GSPC](https://finance.yahoo.com/quote/%5EGSPC); [^OVX](https://finance.yahoo.com/quote/%5EOVX); [BZ=F](https://finance.yahoo.com/quote/BZ%3DF); [CADUSD=X](https://finance.yahoo.com/quote/CADUSD=X); [GC=F](https://finance.yahoo.com/quote/GC%3DF) | S&P 500 weekly log return (`sp500_log_return`; non-stationary index level dropped); CBOE Crude Oil Volatility Index (OVX); Brent front-month futures–spot log basis (`brent_f1_spot_log_basis`, unadjusted, vs local EIA Brent) + `brent_roll_week` roll dummy; CAD/USD weekly log return (`cadusd_log_return`; AUD leg dropped, kept for robustness); gold weekly log return (`gold_return`) | S&P 500, FX & gold from Jan 2006; OVX from May 2007; futures from Aug 2007 | Daily; aggregated to weekly | US equity & options markets; Brent futures; Canadian/Australian dollar; COMEX gold | Equity / volatility / futures / FX / commodity price | Open | Equity-market risk appetite; oil-specific uncertainty distinct from equity VIX (OVX preferred per literature); term-structure/backwardation signals; commodity-exporter FX channel; gold cross-commodity safe-haven linkage | OVX and futures spread only from 2007; commodity FX is a two-currency proxy; **gold canonical source is FRED LBMA `GOLDPMGBD228NLBM` but currently unstable, so the COMEX `GC=F` snapshot is used** (weekly log-returns match, max\|Δ\|≈3e-16) |
| **Other/** — Geopolitical Risk Index (GPR) | Monthly news-based index measuring geopolitical risk from newspaper articles, constructed by Caldara and Iacoviello (2022); column `GPR` in a Stata `.dta` (`gpr`) | [GPR](https://www.matteoiacoviello.com/gpr.htm) | Geopolitical risk index (overall GPR) | Jan 2006–Dec 2025 | Monthly; month-end forward-filled and lagged 1 week to weekly to respect publication delay | Global | News-text aggregate index | Open (auto-download) | Low-frequency proxy for precautionary oil-demand shifts driven by geopolitical tensions, conflict, and sanctions | Monthly frequency; text-based construction may not capture all geopolitical events equally; limited short-term variation |

**M1) Variable Dictionary — per-variable descriptions / 逐变量说明**

> **Prediction target & baseline / 预测目标与基准**：the **sole research target is next-week Brent price** `P_{t+1}` (USD/bbl). The model is *trained* on the weekly log price change `brent_log_return` (a learning-friendly internal target, not a second goal) and the price is reconstructed as `P_hat = P_t·e^r`. Direction and returns are **auxiliary metrics derived from the predicted price** (not separate tasks; the former `brent_direction` column has been removed). **No independent volatility forecasting**; the realized-volatility columns (`brent_vol_4w/12w`) have been **removed** — implied volatility via `ovx`/`vix` already serves as the volatility feature. **Model 0 / 朴素基准**：p(t+1)=p(t)，即预测收益为 0。 / **唯一研究目标 = 下一周 Brent 价格**；训练用对数价格变化 `brent_log_return` 作内部目标、再还原价格；方向/收益率由预测价格派生为辅助评估指标（原 `brent_direction` 列已移除）；**不做独立波动率预测，已删除已实现波动率列 `brent_vol_4w/12w`**（波动率信息由隐含波动率 `ovx`/`vix` 承载）。
> Units / 单位：kb = thousand barrels（千桶）；kbd = thousand barrels/day（千桶/日）。`avail_*` = modality availability flag（模态可用性标记，1/0）。

| Variable | Description (EN) | 描述（中文） |
|---|---|---|
| `brent_price` | Europe Brent spot price (USD/bbl), weekly last | Brent 现货价（美元/桶），周最后值 |
| `wti_price` | WTI Cushing spot price (USD/bbl), weekly last | WTI 库欣现货价，周最后值 |
| `brent_log_return` | Brent weekly log return — **model training target** (internal; price `P_{t+1}` reconstructed as `P_t·e^r`) | Brent 周对数收益率——**模型训练目标**（内部；价格由 `P_t·e^r` 还原） |
| `wti_log_return` | WTI weekly log return (same log convention as Brent) | WTI 周对数收益率（与 Brent 同口径） |
| `brent_wti_spread` | Brent minus WTI price spread (USD/bbl) | Brent 与 WTI 价差（美元/桶） |
| `crude_stocks_excl_spr` | US commercial crude stocks excl. SPR (kb) | 美国商业原油库存（除战略储备）（千桶） |
| `cushing_stocks` | Cushing, OK crude stocks (kb) | 库欣原油库存（千桶） |
| `crude_production` | US weekly crude oil production (kbd) | 美国周度原油产量（千桶/日） |
| `crude_imports` | US weekly crude oil imports (kbd) | 美国周度原油进口（千桶/日） |
| `crude_exports` | US weekly crude oil exports (kbd) | 美国周度原油出口（千桶/日） |
| `refinery_crude_input` | US refinery crude oil inputs (kbd) | 美国炼厂原油投入（千桶/日） |
| `refinery_utilisation` | US refinery utilisation rate (%) | 美国炼厂开工率（%） |
| `gasoline_supplied` | Finished motor gasoline product supplied (kbd; road-fuel demand proxy) | 成品车用汽油表观需求 product supplied（千桶/日；公路燃料需求代理，非严格终端消费） |
| `distillate_supplied` | Distillate fuel oil product supplied (kbd; freight/heating demand proxy) | 馏分油表观需求 product supplied（千桶/日；货运/取暖需求代理，非严格终端消费） |
| `jet_fuel_supplied` | Kerosene-type jet fuel product supplied (kbd; aviation demand proxy) | 航空煤油表观需求 product supplied（千桶/日；航空需求代理，非严格终端消费） |
| `crude_stocks_change` | Weekly first-difference of commercial crude stocks (supply-balance proxy) | 商业原油库存周度一阶差分（供需平衡代理） |
| `cushing_stocks_change` | Weekly first-difference of Cushing stocks | 库欣库存周度一阶差分 |
| `vix` | CBOE VIX close (equity-market implied volatility) | CBOE VIX 收盘（股市隐含波动率） |
| `dollar_index` | Broad trade-weighted US dollar index (DTWEXBGS) | 广义贸易加权美元指数（DTWEXBGS） |
| `treasury_10y` | US 10-year Treasury yield level (%) | 美国 10 年期国债收益率（水平值，%） |
| `fed_funds_rate` | Effective federal funds rate (%) | 联邦基金有效利率（%） |
| `sp500_log_return` | S&P 500 weekly log return (same convention as Brent/gold) | 标普 500 周对数收益（与 Brent/gold 同口径） |
| `ovx` | CBOE Crude Oil Volatility Index (oil-specific implied vol) | CBOE 原油波动率指数（油价专属隐含波动） |
| `gpr` | Geopolitical Risk Index — **daily GPRD** weekly-mean, 1-week lag (Caldara–Iacoviello) | 地缘政治风险指数——**日度 GPRD** 周均值，滞后 1 周（Caldara–Iacoviello） |
| `gold_return` | Gold weekly log return (safe-haven / commodity linkage) | 黄金周对数收益率（避险 / 商品联动） |
| `global_econ_activity` | Kilian index of global real economic activity, 5-week lag | Kilian 全球真实经济活动指数（滞后 5 周） |
| `nonoil_industrial_commodity` | IMF global industrial materials price index, 5-week lag | IMF 全球工业原料价格指数（滞后 5 周） |
| `brent_f1_spot_log_basis` | Brent front-month futures − spot log **basis** (not a pure term-structure spread; unadjusted BZ=F) | Brent 前月期货−现货对数**基差**（非纯期限结构；未 back-adjust） |
| `brent_roll_week` | Contract-roll week dummy (1 = last W-FRI of month ≈ ICE Brent roll) | 换月周哑变量（每月最后周五 ≈ ICE Brent 换月） |
| `cadusd_log_return` | CAD/USD weekly log return (commodity currency, most oil-linked; AUD leg dropped) | CAD/USD 周对数收益（油价敏感商品货币；AUD 腿已删，留稳健性） |
| `dgs10_change` | Weekly first-difference of 10Y Treasury yield (rate change / holding cost) | 10 年期国债收益率周度一阶差分（利率变化 / 持有成本） |
| `avail_market` | Availability flag: market price present that week (1/0) | 可用性标记：当周有市场价（1/0） |
| `avail_eia_weekly` | Availability flag: EIA weekly fundamentals present (1/0) | 可用性标记：当周有 EIA 周度基本面（1/0） |
| `avail_sp500` | Availability flag: S&P 500 present (1/0) | 可用性标记：当周有标普 500（1/0） |
| `avail_dollar_index` | Availability flag: dollar index present (1/0) | 可用性标记：当周有美元指数（1/0） |

**M2) Remote Sensing Variables**

**Dual-channel Earth-observation design** over 11 global oil-infrastructure AOIs (centre points in `aoi_oil_infrastructure.csv`; per-site profiles in `aoi_oil_infrastructure_sites.md`). **Channel A (image representation — the methodological core)** exports monthly cloud-masked Sentinel-2 6-band patches per site and feeds them to a *frozen* pretrained EO foundation model (Prithvi-EO-2.0 / SatMAE) for image embeddings — imagery is no longer flattened into hand-crafted indices. **Channel B (mechanism variables — economic interpretation, auxiliary)** retains Sentinel-2 spectral indices and VIIRS nighttime-light statistics as site-activity / information-availability proxies. The patch window is **2019-01 – 2026-06** (aligned to the standardised M0–M4 comparison window); Channel-B series run longer (S2 indices from 2017-04, VIIRS from 2014-01). Observations are **asynchronously aligned by real observation/release date (no constant-value forward-fill)**, each carrying `days_since_obs / cloud_fraction / valid_mask` for explicit missing-modality modelling. All obtained free via Google Earth Engine Code Editor (no Python key); scripts live under `03_data/raw/02_sentinel2/Channel A/` and `Channel B/`.

| Dataset | Summary | URL | Variables | Time Range | Frequency | Coverage | Data Type | Access | Potential Use | Limitations |
|---------|---------|-----|-----------|------------|-----------|----------|-----------|--------|---------------|-------------|
| **Channel A** — Copernicus Sentinel-2 SR 6-band patches (foundation-model-ready) | Monthly SCL-cloud-masked median composites of 6 reflectance bands (B2/B3/B4/B8A/B11/B12, matching the HLS 6-band set used by Prithvi-EO), one GeoTIFF per (site, month) at 10 m in per-site UTM, plus a manifest of valid-scene counts and cloud stats | [Google Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) | One 6-band image patch per (site, month); per-site differentiated patch size (default port 6.4 km / refinery 5.12 km / terminal 2.56 km, with visual-check overrides 3.2 km for Fujairah/Kharg/Yanbu and 1.6 km for Basra; actual half-size from `patch_half_m`). Manifest fields `n_scenes`, `exported`, `min_cloud`, `mean_cloud`, `patch_px`, `crs`. Fed to a frozen EO model → image embedding + `days_since_obs`/`cloud_fraction`/`valid_mask` | Jan 2019–Jun 2026 | Monthly (irregular, cloud-limited); median composite, empty months skipped | 11 oil-infrastructure AOIs @ 10 m, per-site UTM | Remote sensing (multispectral imagery) | Open (Google Earth Engine) | Learned image representation of site-level oil activity (tank farms, terminals, berths, refinery units) via a frozen EO foundation model + temporal/site attention — the methodological contribution of M2 | Cloudy/empty months yield no patch (`n_scenes`=0, skipped); 10 m insufficient for vessel-level or single-tank detection; embeddings not directly interpretable; shorter window than M1; needs leakage-safe async alignment |
| **Channel B** — Copernicus Sentinel-2 SR spectral indices + cloud probability | Monthly per-AOI mean and std of NDVI/NDWI/NDBI/BSI from cloud-masked S2 SR (s2cloudless + SCL), plus mean cloud probability and a clear-observation count | [Google Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) | Per AOI: `NDVI`, `NDWI`, `NDBI`, `BSI` (each with `_std`), `cloud_probability`, `valid_obs_count`. Output `sentinel2_oil_sites_monthly_indices_201704_202512_11aoi.csv` | Apr 2017–Dec 2025 | Monthly (async-aligned by observation date; no constant-value ffill) | 11 oil-infrastructure AOIs, 5 km circular buffer, 10 m resolution | Remote sensing (optical indices) | Open (Google Earth Engine) | Mechanism proxies for vegetation/water/built-up/bare-soil surface state around oil sites; cloud probability and valid-obs count as information-availability / data-gap indicators | Indices are coarse activity proxies, not throughput; cloud cover reduces usable observations; auxiliary channel only |
| **Channel B** — NOAA/NASA VIIRS DNB monthly nighttime lights | Monthly per-AOI radiance statistics (`avg_rad` mean/max/std) and cloud-free coverage count (`cf_cvg`), from which a within-site expanding-window z-score anomaly is derived downstream | [Google Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG) | Per AOI: `ntl_avg_rad_mean`, `ntl_avg_rad_max`, `ntl_avg_rad_stddev`, `ntl_cf_cvg_mean`; derived `ntl_anomaly` (expanding-window z-score, past-only, min 12 months) and `ntl_valid_obs_count`. Output `viirs_oil_sites_monthly_nightlights_201401_202512_11aoi.csv` | Jan 2014–Dec 2025 | Monthly (async-aligned; no constant-value ffill) | 11 oil-infrastructure AOIs, 5 km circular buffer, ~500 m resolution | Remote sensing (night lights) | Open (Google Earth Engine) | Proxy for anomalous nocturnal activity at ports/terminals relative to site-specific history; cloud-free count for data-quality control | Night lights are indirect proxies (NTL↔tanker Rs≈−0.07); affected by urban spillover, gas flaring, spatial bleed; requires within-site normalisation; auxiliary channel only |

**M2) Variable Dictionary — per-variable descriptions / 逐变量说明**

> Dual-channel design / 双通道：**Channel A** = foundation-model image embeddings (representation learning, core); **Channel B** = mechanism variables repeated across the 11 AOIs with suffix `_{aoi}` (economic interpretation, auxiliary). / **通道 A** = EO 大模型影像嵌入（表示学习，核心）；**通道 B** = 按 11 个 AOI 重复、后缀 `_{aoi}` 的机制变量（经济解释，辅助）。
> `{aoi}` ∈ Rotterdam, Fujairah, RasTanura, Jurong, Houston, NingboZhoushan, Jamnagar, Basra, Ulsan, Kharg, Yanbu (P001–P011). / `{aoi}` 为 11 个站点代码（P001–P011）。

| Variable (pattern) | Channel | Description (EN) | 描述（中文） |
|---|---|---|---|
| `s2_patch_{aoi}_{YYYY_MM}` | A | Monthly cloud-masked 6-band (B2/B3/B4/B8A/B11/B12) Sentinel-2 GeoTIFF patch per AOI; input to the frozen EO foundation model | 每 AOI 每月无云 6 波段 S2 影像 patch，喂冻结 EO 大模型 |
| `image_embedding_{aoi}` | A | Per-patch feature vector from the frozen EO encoder (Prithvi-EO-2.0 / SatMAE), pooled by temporal + site attention | 冻结 EO 编码器输出的 patch 特征向量（经时间/站点注意力池化） |
| `days_since_obs_{aoi}` | A | Age (days) of the latest valid patch at prediction time (asynchronous-alignment / staleness signal) | 预测时点距最近有效观测的天数（异步对齐 / 时效信号） |
| `cloud_fraction_{aoi}` | A | Fraction of the patch masked by cloud/shadow (per-observation quality weight) | patch 被云/阴影掩膜的比例（单次观测质量权重） |
| `valid_mask_{aoi}` / `modality_mask` | A | 1 if a valid patch / modality is available this step, else 0 (explicit missing-modality modelling) | 当步是否有有效 patch / 模态（1/0，显式建模缺失） |
| `n_scenes_{aoi}` | A | Manifest: count of cloud-filtered S2 scenes in the (site, month) composite (0 = month skipped) | 清单：该站点该月用于合成的可用 S2 影像数（0 表示跳过） |
| `ntl_avg_rad_mean_{aoi}` | B | VIIRS mean nighttime radiance over the AOI buffer (nW/cm²/sr) | AOI 缓冲区 VIIRS 平均夜间辐亮度 |
| `ntl_avg_rad_max_{aoi}` / `_stddev_{aoi}` | B | Max / std of VIIRS radiance within the AOI | AOI 内 VIIRS 辐亮度最大值 / 标准差 |
| `ntl_anomaly_{aoi}` | B (derived) | Within-site expanding-window z-score of monthly mean radiance (past-only, min 12 months) | 站点扩展窗 z-score（仅用历史，≥12 月） |
| `ntl_cf_cvg_mean_{aoi}` / `ntl_valid_obs_count_{aoi}` | B | VIIRS cloud-free observation count (data-quality / completeness control) | VIIRS 无云观测计数（数据质量 / 完整性控制） |
| `s2_ndvi_{aoi}` / `s2_ndwi_{aoi}` / `s2_ndbi_{aoi}` / `s2_bsi_{aoi}` | B | Monthly mean Sentinel-2 spectral indices (vegetation / water / built-up / bare-soil) per AOI; each also has a `_std` | 每 AOI 月度 S2 光谱指数均值（植被/水体/建成区/裸土，各含 `_std`） |
| `s2_cloud_probability_{aoi}` | B | Mean s2cloudless cloud probability over the AOI (information-gap / uncertainty proxy) | AOI 平均云概率（信息缺口 / 不确定性代理） |
| `s2_valid_obs_count_{aoi}` | B | Number of clear Sentinel-2 observations in the monthly composite | 参与月度合成的晴空 S2 观测数 |

**M2) Channel B weekly feature table — column dictionary / 周频建模表列字典**

> Built by `03_data/processed/M2/py/build_m2_weekly.py` (step B1) from the two raw monthly tables above; this is the **modelling-ready weekly (W-FRI) table** `processed/M2/outputs/m2_weekly_features.csv` actually fed to the M2 ablations — distinct from the raw monthly variables listed above. / 由 `build_m2_weekly.py`（B1）从上方两张原始月度表构建，是真正进入 M2 消融的**周频建模表**，区别于上方原始月度变量。
> Window 2019–2026 (365 W-FRI weeks); **154 features** = 5 indicators × 11 AOIs × (level + anom) + 2 modalities × 11 AOIs × (age + avail). `{idx}` ∈ NDVI/NDWI/NDBI/BSI/NTL; `{aoi}` ∈ Houston, NingboZhoushan, Rotterdam, Jamnagar, Jurong, Ulsan, Basra, Fujairah, Kharg, RasTanura, Yanbu. / 窗口 2019–2026（365 周），154 特征 = 5 指标 ×11 站 ×(level+anom) + 2 模态 ×11 站 ×(age+avail)。

| Column (pattern) | Form | Description (EN) | 描述（中文） |
|---|---|---|---|
| `{idx}_{aoi}` | level | Most-recent already-released monthly value, as-of aligned to that Friday (repeats within a month) | 截至该周五最近已发布的月度值（月内重复） |
| `{idx}_anom_{aoi}` | anom | Within-site standardized anomaly: de-seasonalised (expanding month-of-year climatology) then expanding z-score, past-only, min 12 months — **the main modelling form** | 站点标准化距平：先去季节再 expanding z-score，仅用历史、≥12 月——**入模主力** |
| `s2_age_days_{aoi}` / `ntl_age_days_{aoi}` | meta | `days_since_obs`: age in days of the latest valid obs at that Friday (staleness / time-gap signal) | 距最近有效观测的天数（时效 / time-gap 信号） |
| `s2_avail_{aoi}` / `ntl_avail_{aoi}` | meta | `modality_mask`: 1 if a fresh-enough valid obs exists (age ≤ 100 d), else 0 | 模态可用标记：存在足够新鲜的有效观测则 1 |

**Long (tidy) companion** `m2_weekly_long.csv` (one row per week × site × indicator) additionally carries: `mom` (month-on-month first difference), `observation_date`, `valid_mask` (a valid obs found at all), `valid_obs_count` (S2 clear-obs count / VIIRS cloud-free count). / 长表每行 = 周 × 站 × 指标，另含 `mom`（月差分）、`observation_date`、`valid_mask`、`valid_obs_count`。

**Alignment & leakage / 对齐与防泄漏**: monthly→weekly via **as-of join** on availability date = month_end + 15 d (conservative release lag); **no constant-value forward-fill** (value repeats but `age` increments and jumps only on a new release). `cloud_probability` is **not a model feature** (audit / filtering only). All anomaly statistics are expanding / past-only. / 月→周用可得日（月末+15 天）**as-of join**，非哑 ffill（值重复但 `age` 递增、仅新发布时跳变）；`cloud_probability` 不入模；距平统计全程仅用历史。

**M3) Shipping & Port Activity Variables**

119 candidate features derived from AIS-based maritime datasets, focused on tanker-specific indicators: transit flow intensity, deadweight-tonnage (DWT) capacity weighting, average vessel size, chokepoint-level disaggregation, export–import directional asymmetry, and congestion/dwell proxies. Covers 6 major oil-related maritime chokepoints (Strait of Hormuz, Suez Canal, Strait of Malacca, Bab el-Mandeb, Panama Canal, Cape of Good Hope) and 14 key oil-tanker port hubs.

| Dataset | Summary | URL | Variables | Time Range | Frequency | Coverage | Data Type | Access | Potential Use | Limitations |
|---------|---------|-----|-----------|------------|-----------|----------|-----------|--------|---------------|-------------|
| IMF PortWatch (Chokepoint Transits + Port Activity) | AIS-derived daily vessel transits through maritime chokepoints by vessel type and capacity, plus port-level tanker import/export tonnage estimates at 14 major oil hubs | [IMF PortWatch](https://portwatch.imf.org/) | Chokepoint-level (6 chokepoints × 9 indicators): tanker transit count, tanker capacity, tanker share of total traffic, tanker capacity share, average tanker size, week-on-week change, 4-week moving average, total vessel count, total capacity; global aggregates. Port-level (7 indicators): export hub volume, import hub volume, net export–import balance, asymmetry ratio, log ratio, 4-week smoothed asymmetry, week-on-week export change. 64 features total | Jan 2019–Dec 2025 | Daily; aggregated to weekly sums | 6 maritime chokepoints (Hormuz, Suez, Malacca, Bab el-Mandeb, Panama, Cape) + 14 tanker hubs (9 export, 5 import/refining) | Shipping / port activity | Open (ArcGIS FeatureServer) | Tanker traffic-intensity proxy for seaborne crude oil flow; chokepoint disruption detection; directional export–import asymmetry reflecting loading vs unloading imbalances | Chokepoint transits are a coarse proxy, not exact oil trade volumes; chokepoint data does not distinguish transit direction; port tonnage estimated from AIS vessel draft rather than direct cargo measurement; data available only from 2019 |
| Global Fishing Watch Vessel Presence (4Wings API) | AIS-derived monthly vessel-presence hours and vessel counts by type within chokepoint polygons, providing vessel dwell-time and congestion indicators | [GFW APIs](https://globalfishingwatch.org/our-apis/) | Per chokepoint (6 chokepoints × 9 indicators): total presence hours, total vessel count, cargo-vessel hours, bunker-vessel hours, other-vessel hours, non-tanker hours, other-vessel share, month-on-month change, dwell hours per vessel (congestion proxy); global aggregate. 55 features total | Jan 2012–Dec 2025 | Monthly; forward-filled to weekly | 6 chokepoint polygons | Shipping / AIS presence | Open (free API token required) | Congestion and dwell-time proxy based on average hours per vessel; longer historical coverage than PortWatch (from 2012); vessel-type breakdown for traffic composition analysis | Monthly granularity limits detection of short-term disruptions; no dedicated anchorage-waiting or queue-length data; vessel-type categories are broad and not purely oil-tanker specific; tanker filtering is approximate |

**M3) Variable Dictionary — per-variable descriptions / 逐变量说明**

> Variables follow naming patterns expanded across 6 oil chokepoints (`{choke}` ∈ Hormuz, Suez, Malacca, BabelMandeb, Panama, Cape) and export/import hub baskets (~119 features total). / 变量按 6 个油运咽喉（`{choke}`：Hormuz、Suez、Malacca、BabelMandeb、Panama、Cape）与出/进口枢纽篮子展开（共约 119 个特征）。
> Prefix legend / 前缀：`pw_` = IMF PortWatch；`gfw_` = Global Fishing Watch；`emodnet_` = EMODnet vessel density。Suffixes / 后缀：`_wow_pct` 周环比、`_mom_pct` 月环比、`_4w_ma` 4 周均值、`_sum` 全局汇总。

| Variable (pattern) | Description (EN) | 描述（中文） |
|---|---|---|
| `pw_{choke}_n_tanker` | PortWatch weekly tanker transit count through the chokepoint | PortWatch 周度油轮过境数 |
| `pw_{choke}_capacity_tanker` | Weekly tanker transit capacity (DWT-weighted) | 周度油轮过境运力（DWT 加权） |
| `pw_{choke}_n_total` / `_capacity` | Total vessels / total capacity (all types) | 总船舶数 / 总运力（全船型） |
| `pw_{choke}_tanker_share` | Tanker share of total transit count | 油轮占总过境数比例 |
| `pw_{choke}_tanker_cap_share` | Tanker share of total transit capacity | 油轮占总运力比例 |
| `pw_{choke}_avg_tanker_size` | Average tanker size = tanker capacity / tanker count | 平均油轮船型 = 油轮运力 / 油轮数 |
| `pw_{choke}_n_tanker_wow_pct` | Week-on-week % change of tanker count | 油轮数周环比变化（%） |
| `pw_{choke}_capacity_tanker_4w_ma` | 4-week moving average of tanker capacity | 油轮运力 4 周移动平均 |
| `pw_all_n_tanker_sum` / `_n_total_sum` / `_tanker_share` | Global aggregates across all chokepoints | 全咽喉全局汇总 |
| `pw_exp_hubs_export_vol` | Tanker export tonnage at export-hub basket | 出口枢纽篮子油轮出口吨位 |
| `pw_imp_hubs_import_vol` | Tanker import tonnage at import-hub basket | 进口枢纽篮子油轮进口吨位 |
| `pw_tanker_exp_imp_net` | Net export − import tonnage | 出口−进口净吨位 |
| `pw_tanker_exp_imp_asym` / `_log_ratio` | Export/import asymmetry ratio / log ratio | 出进口不对称比 / 对数比 |
| `pw_tanker_exp_imp_asym_4w_ma` | 4-week smoothed export–import asymmetry | 出进口不对称 4 周平滑 |
| `pw_exp_hubs_export_vol_wow_pct` | Week-on-week % change of export volume | 出口量周环比变化（%） |
| `gfw_{choke}_total_hours` | GFW AIS total vessel-presence hours in the chokepoint polygon | GFW AIS 咽喉多边形内总在场小时 |
| `gfw_{choke}_total_vessels` | Total vessel count (AIS presence) | 总船舶数（AIS 在场） |
| `gfw_{choke}_cargo_hours` / `_bunker_hours` / `_other_hours` / `_nontanker_hours` | Presence hours by vessel type | 按船型在场小时 |
| `gfw_{choke}_other_share` | Other-vessel share of presence hours | 其他船型在场小时占比 |
| `gfw_{choke}_total_hours_mom_pct` | Month-on-month % change of presence hours | 在场小时月环比变化（%） |
| `gfw_{choke}_dwell_hours_per_vessel` | Dwell hours per vessel = total hours / vessels (congestion proxy) | 单船停留时长 = 总小时 / 船数（拥堵代理） |
| `gfw_all_total_hours_sum` | Global aggregate presence hours across chokepoints | 全咽喉全局汇总在场小时 |
| `emodnet_{node}_vessel_density` | EMODnet monthly vessel-density raster aggregated over node/chokepoint polygon (regional trade-flow intensity) | EMODnet 月度船舶密度栅格在节点/咽喉多边形上的区域统计（区域贸易流强度） |